In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, confusion_matrix, classification_report
)
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Section 2 — Load & Inspect the Real Survey Data

In [2]:
DATA_PATH = "data1001_survey_data_2025_S2-1.csv"

raw = pd.read_csv(DATA_PATH)

# Keep only rows where consent was given
raw = raw[raw["consent"].str.contains("consent to take part", na=False, case=False)].copy()

print(f"Rows after consent filter: {len(raw)}")
print(f"Columns: {list(raw.columns)}")
raw.head(3)

Rows after consent filter: 2955
Columns: ['cohort', 'consent', 'age', 'gender', 'country_of_birth', 'country_of_birth_5_TEXT', 'hours_work', 'social_media_use', 'rent', 'friends_count', 'stress', 'highest_speed', 'relationship_status', 'dates', 'standard_drinks', 'countries', 'drug_use_q', 'drug_use_ans', 'student_type', 'mainstream_advanced', 'semesters', 'commute', 'data_interest', 'mark_goal', 'hours_studying', 'lecture_mode', 'study_type', 'learner_style']


,cohort,consent,age,gender,country_of_birth,country_of_birth_5_TEXT,hours_work,social_media_use,rent,friends_count,...,student_type,mainstream_advanced,semesters,commute,data_interest,mark_goal,hours_studying,lecture_mode,study_type,learner_style
0,2024S2,I DO NOT consent to take part in the study,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024S2,I consent to take part in the study,18.0,Female,Australia,NaN,40.0,4.0,400.0,2.0,...,International,DATA1001,4.0,70.0,9.0,50.0,5.0,Live in the Lecture Theatre,I work steadily all semester,Style 1
2,2024S2,I consent to take part in the study,19.0,Female,Other Please Specify,NaN,40.0,200.0,200.8,0.0,...,International,DATA1901,8.3,5.0,4.0,50.0,25.0,Other,It changes depending on the subject,Style 3


## Section 3 — Feature Engineering & Stress Score Binning

In [3]:
# ── Column definitions ────────────────────────────────────────────────────────
NUMERIC_COLS = [
    "age", "hours_work", "social_media_use", "rent",
    "friends_count", "highest_speed", "dates", "standard_drinks",
    "countries", "semesters", "commute", "data_interest",
    "mark_goal", "hours_studying",
]
CATEGORICAL_COLS = [
    "gender", "relationship_status", "drug_use_ans",
    "student_type", "mainstream_advanced", "lecture_mode",
    "study_type", "learner_style",
]
TARGET = "stress"

# ── Build working dataframe ───────────────────────────────────────────────────
cols_needed = NUMERIC_COLS + CATEGORICAL_COLS + [TARGET]
df = raw[cols_needed].copy()

# ── Step 1: Outlier clipping — 1st to 99th percentile per numeric column ─────
for col in NUMERIC_COLS:
    lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
    df[col] = df[col].clip(lo, hi)

# ── Step 2: Impute missing values ─────────────────────────────────────────────
#    Numeric → median (robust to skew); Categorical → mode
for col in NUMERIC_COLS:
    df[col] = df[col].fillna(df[col].median())
for col in CATEGORICAL_COLS:
    mode_val = df[col].mode()
    df[col] = df[col].fillna(mode_val[0] if not mode_val.empty else "Unknown")

# ── Step 3: Derived / engineered features ────────────────────────────────────
# Financial pressure: rent per worked hour (higher → more financial burden)
df["financial_pressure"] = (
    df["rent"] / df["hours_work"].replace(0, np.nan)
).fillna(df["rent"])

# Work-to-study ratio: balance between earning and learning obligations
df["work_study_ratio"] = (
    df["hours_work"] / df["hours_studying"].replace(0, np.nan)
).fillna(df["hours_work"])

# Social engagement: friends relative to social media time (quality vs quantity)
df["social_engagement"] = (
    df["friends_count"] / df["social_media_use"].replace(0, np.nan)
).fillna(df["friends_count"])

# Clip derived features to 99th percentile to suppress extreme ratios
for col in ["financial_pressure", "work_study_ratio", "social_engagement"]:
    df[col] = df[col].clip(0, df[col].quantile(0.99))

DERIVED_COLS = ["financial_pressure", "work_study_ratio", "social_engagement"]

# ── Step 4: Label-encode categoricals — FOR EDA / correlation plots ONLY ─────
#    The modeling pipeline (Section 5) uses OneHotEncoding instead.
le_dict = {}
df_eda = df.copy()
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df_eda[col] = le.fit_transform(df_eda[col].astype(str))
    le_dict[col] = le

# ── Step 5: Drop rows with missing target ─────────────────────────────────────
df     = df.dropna(subset=[TARGET])
df_eda = df_eda.loc[df.index]

# ── Step 6: Bin continuous stress score into 3 categories ─────────────────────
def bin_stress(score):
    """Map a 0–10 stress score → Low / Average / High."""
    if score <= 3:   return "Low (1-3)"
    elif score <= 6: return "Average (4-6)"
    else:            return "High (7-10)"

df["stress_category"]     = df[TARGET].apply(bin_stress)
df_eda["stress_category"] = df_eda[TARGET].apply(bin_stress)
CATEGORY_ORDER = ["Low (1-3)", "Average (4-6)", "High (7-10)"]

print(f"Dataset shape     : {df.shape}")
print(f"Numeric features  : {NUMERIC_COLS}")
print(f"Derived features  : {DERIVED_COLS}")
print(f"Categorical feat. : {CATEGORICAL_COLS}")
print("\nStress category distribution:")
print(df["stress_category"].value_counts().reindex(CATEGORY_ORDER))
df.head(4)

Dataset shape     : (2842, 27)
Numeric features  : ['age', 'hours_work', 'social_media_use', 'rent', 'friends_count', 'highest_speed', 'dates', 'standard_drinks', 'countries', 'semesters', 'commute', 'data_interest', 'mark_goal', 'hours_studying']
Derived features  : ['financial_pressure', 'work_study_ratio', 'social_engagement']
Categorical feat. : ['gender', 'relationship_status', 'drug_use_ans', 'student_type', 'mainstream_advanced', 'lecture_mode', 'study_type', 'learner_style']

Stress category distribution:
stress_category
Low (1-3)         773
Average (4-6)    1195
High (7-10)       874
Name: count, dtype: int64


,age,hours_work,social_media_use,rent,friends_count,highest_speed,dates,standard_drinks,countries,semesters,...,student_type,mainstream_advanced,lecture_mode,study_type,learner_style,stress,financial_pressure,work_study_ratio,social_engagement,stress_category
1,18.00,40.0,4.0,400.0,2.0,150.0,0.0,6.0,6.0,4.0,...,International,DATA1001,Live in the Lecture Theatre,I work steadily all semester,Style 1,10.0,10.000000,8.000000,0.500000,High (7-10)
2,19.00,40.0,22.5,200.8,0.0,0.0,0.0,4.3,6.0,8.3,...,International,DATA1901,Other,It changes depending on the subject,Style 3,1.0,5.020000,1.600000,0.000000,Low (1-3)
3,33.59,38.5,2.0,1000.0,100.0,140.0,0.0,0.0,6.0,0.0,...,Domestic,DATA1001,Other,I work steadily all semester,Style 1,5.0,25.974026,3.850000,32.752688,Average (4-6)
4,17.00,10.0,10.0,500.0,1.0,200.0,0.0,0.0,6.0,5.0,...,Domestic,DATA1001,Live in the Lecture Theatre,I work steadily all semester,Style 2,7.0,50.000000,3.333333,0.100000,High (7-10)


## Section 5. Data Processing

In [4]:
# ── Feature sets for modeling ─────────────────────────────────────────────────
ALL_NUMERIC = NUMERIC_COLS + DERIVED_COLS   # 14 raw + 3 engineered = 17
ALL_CATS    = CATEGORICAL_COLS              # 8 nominal columns

# ── Rebuild X with: numeric from df (clipped+derived), strings from raw ───────
#    df already has outlier-clipped numerics and derived features computed.
#    We restore original string categoricals from raw so OHE works correctly
#    (df has label-integers for EDA; pipeline needs the real strings).
X_num = df[ALL_NUMERIC].copy()
X_cat = raw.loc[df.index, ALL_CATS].fillna("Unknown")
X_raw = pd.concat([X_num, X_cat], axis=1)

y_continuous = df[TARGET]
y_category   = df["stress_category"]

# ── Preprocessing pipeline ────────────────────────────────────────────────────
#    Numeric  : median imputation → StandardScaler
#    Categorical: most-frequent imputation → OneHotEncoder (no ordinal assumption)
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, ALL_NUMERIC),
    ("cat", categorical_transformer, ALL_CATS),
])

# ── 70 / 15 / 15 train-test split (stratified on stress category) ─────────────────
X_raw_train_val, X_raw_test, y_train_val, y_test, ycat_train_val, ycat_test = train_test_split(
    X_raw, y_continuous, y_category,
    test_size=0.85, random_state=42, stratify=y_category
)

val_size = 0.15/0.85

X_raw_train, X_raw_val, y_train, y_val, ycat_train, ycat_val = train_test_split(
    X_raw_train_val, y_train_val, ycat_train_val,
    test_size=val_size, random_state=42, stratify=ycat_train_val
)


# ── Fit on train set, transform both ─────────────────────────────────────────
X_train = preprocessor.fit_transform(X_raw_train)
X_val   = preprocessor.transform(X_raw_val)
X_test  = preprocessor.transform(X_raw_test)

print(f"Training samples             : {len(X_train)}")
print(f"Training samples             : {len(X_val)}")
print(f"Test samples                 : {len(X_test)}")
print(f"Raw feature count            : {len(ALL_NUMERIC + ALL_CATS)}")
print(f"Features after OHE expansion : {X_train.shape[1]}")
print("\nPreprocessing pipeline:")
print("  Numeric  → Median Imputer → StandardScaler")
print("  Categorical → Mode Imputer → OneHotEncoder")

Training samples             : 350
Training samples             : 76
Test samples                 : 2416
Raw feature count            : 25
Features after OHE expansion : 43

Preprocessing pipeline:
  Numeric  → Median Imputer → StandardScaler
  Categorical → Mode Imputer → OneHotEncoder


## Section 6 — Model Training

In [5]:
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import PolynomialFeatures

In [6]:
alpha_n = 10
exponents = np.arange(-1, -alpha_n - 1, -1) 
alpha = 10**exponents.astype(float)
print(alpha)

[1.e-01 1.e-02 1.e-03 1.e-04 1.e-05 1.e-06 1.e-07 1.e-08 1.e-09 1.e-10]


In [11]:
orders = np.arange(1, 15, 2)
orders

array([1, 3, 5])

In [17]:
models = []
train_rmse = []
valid_rmse = []
model_order = []
model_alpha = []
for a in alpha:
    for o in orders:
        #Data Pre-processing
        X1 = X_train
        X2 = X_val
        poly = PolynomialFeatures(degree=o, include_bias=True)
        train_poly_X = poly.fit_transform(X1)
        val_poly_X = poly.fit_transform(X2)
        
        #Modelling
        model = Lasso(alpha = a, max_iter = 25,random_state=42)
        model.fit(train_poly_X,y_train)
        
        
        #Comparison
        y_pred_train = model.predict(train_poly_X)
        y_pred_val = model.predict(val_poly_X)
        training_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
        val_rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
        
        #Appening to lists
        models.append(model)
        model_order.append(o)
        model_alpha.append(a)
        train_rmse.append(training_rmse)
        valid_rmse.append(val_rmse)

comparison = {'order': model_order, 'alpha': model_alpha,'training rmse': train_rmse, 'validation rmse': valid_rmse}


MemoryError: Unable to allocate 4.47 GiB for an array with shape (350, 1712304) and data type float64

In [10]:
comparison = pd.DataFrame(comparison)
print(comparison)

    order         alpha  training rmse  validation rmse
0       1  1.000000e-01   2.276766e+00         2.137694
1       3  1.000000e-01   1.691826e+00         2.317836
2       1  1.000000e-02   2.202664e+00         2.179898
3       3  1.000000e-02   5.597244e-01         2.886626
4       1  1.000000e-03   2.186061e+00         2.312517
5       3  1.000000e-03   9.177335e-02         2.921882
6       1  1.000000e-04   2.185848e+00         2.333763
7       3  1.000000e-04   1.339431e-02         3.271310
8       1  1.000000e-05   2.185845e+00         2.335958
9       3  1.000000e-05   1.950306e-03         3.056658
10      1  1.000000e-06   2.185845e+00         2.336176
11      3  1.000000e-06   2.257213e-04         3.135253
12      1  1.000000e-07   2.185845e+00         2.336199
13      3  1.000000e-07   2.321565e-05         3.164825
14      1  1.000000e-08   2.185845e+00         2.336202
15      3  1.000000e-08   6.332496e-06        13.524666
16      1  1.000000e-09   2.185845e+00         2

In [10]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

#Tests for multicollinearity for optimization function selection
X = add_constant(X_num)

def calc_vif(X):
    vif = pd.DataFrame()
    vif["variables"] = X.columns
    vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif

# Calculate and print VIF values
vif_results = calc_vif(X)
print(vif_results)

#no collinearity implies Ridge might be worse than Lasso for optimization

             variables         VIF
0                const  130.470026
1                  age    1.489044
2           hours_work    3.281397
3     social_media_use    1.346797
4                 rent    3.132723
5        friends_count    2.216892
6        highest_speed    1.080020
7                dates    1.057486
8      standard_drinks    1.207068
9            countries    1.170590
10           semesters    1.439250
11             commute    1.321779
12       data_interest    1.155386
13           mark_goal    1.074007
14      hours_studying    1.416069
15  financial_pressure    3.102185
16    work_study_ratio    2.979375
17   social_engagement    2.406080
